# Outage and weather dual-axis exploration

This notebook loads the event-level POUS data, the hourly ERA5 weather export, and a small subset of the outage time series. It then plots outage fraction and wind gust on dual axes for a few Florida events.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from pathlib import Path
from matplotlib.dates import AutoDateLocator, ConciseDateFormatter

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

POUS_PATH = Path(r"C:\\Users\\teaching\\Downloads\\outage-recovery-forecasting\\data_raw\\POUS.csv")
TIMESERIES_PATH = Path(r"C:\\Users\\teaching\\Downloads\\outage-recovery-forecasting\\data_raw\\timeseries.pq")
WEATHER_PATH = Path(r"C:\\Users\\teaching\\Downloads\\outage-recovery-forecasting\\data_raw\\toy_tft_florida_event_weather_era5_hourly.csv")

print("POUS:", POUS_PATH)
print("Timeseries:", TIMESERIES_PATH)
print("Weather:", WEATHER_PATH)


POUS: C:\Users\teaching\Downloads\outage-recovery-forecasting\data_raw\POUS.csv
Timeseries: C:\Users\teaching\Downloads\outage-recovery-forecasting\data_raw\timeseries.pq
Weather: C:\Users\teaching\Downloads\outage-recovery-forecasting\data_raw\toy_tft_florida_event_weather_era5_hourly.csv


In [2]:
# Load the event-level POUS table and the weather export.
pous = pd.read_csv(POUS_PATH)
pous["CountyFIPS"] = pous["CountyFIPS"].astype(str).str.zfill(5)
pous["event_start"] = pd.to_datetime(pous["event_start"])

weather = pd.read_csv(WEATHER_PATH)
weather["geoid"] = weather["geoid"].astype(str).str.zfill(5)
weather["datetime"] = pd.to_datetime(weather["datetime"]).dt.floor("H")
weather["event_start"] = pd.to_datetime(weather["event_start"])

print("POUS shape:", pous.shape)
print("Weather shape:", weather.shape)
print("POUS columns:")
print(list(pous.columns))
print("Weather columns:")
print(list(weather.columns))


POUS shape: (1040, 13)
Weather shape: (1982, 14)
POUS columns:
['event_start', 'CountyFIPS', 'county_pop', 'pre_outage_tracked_customers', 'days_since_data_start', 'duration_hours', 'n_periods', 'integral', 'pop_hours_supply_lost', 'storm', 'duration_days', 'year', 'month']
Weather columns:
['system:index', 'county', 'datetime', 'duration_hours', 'event_id', 'event_start', 'geoid', 'gust_mps', 'precip_mm', 'pressure_hpa', 'storm', 'temp_c', 'wind_speed_mps', '.geo']


C:\Users\teaching\AppData\Local\Temp\ipykernel_31276\3646064151.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  weather["datetime"] = pd.to_datetime(weather["datetime"]).dt.floor("H")


In [3]:
# Recover the event list from the weather export so the plotting notebook stays aligned
# with the exact event windows used in Earth Engine.
event_meta = (
    weather[["event_id", "storm", "geoid", "event_start", "duration_hours"]]
    .drop_duplicates()
    .sort_values(["event_start", "geoid"])
    .reset_index(drop=True)
)

N_EVENTS = 3
selected_events = event_meta.head(N_EVENTS).copy()
selected_events["window_start"] = selected_events["event_start"] - pd.Timedelta(hours=12)
selected_events["window_end"] = selected_events["event_start"] + pd.to_timedelta(selected_events["duration_hours"], unit="h") + pd.Timedelta(hours=12)

print(selected_events)


    event_id          storm  geoid         event_start  duration_hours        window_start          window_end
0  event_002  2017242N16333  12067 2017-09-09 20:00:00             191 2017-09-09 08:00:00 2017-09-18 07:00:00
1  event_001  2017242N16333  12075 2017-09-09 22:00:00             190 2017-09-09 10:00:00 2017-09-18 08:00:00
2  event_003  2017242N16333  12079 2017-09-10 02:00:00             124 2017-09-09 14:00:00 2017-09-15 18:00:00


In [7]:
# Helper: read only the needed outage rows from the huge parquet file.
def load_outage_window(timeseries_path, county_fips, window_start, window_end):
    import pandas as pd
    import pyarrow.dataset as ds

    county_fips = str(county_fips).zfill(5)
    window_start = pd.to_datetime(window_start)
    window_end = pd.to_datetime(window_end)

    dataset = ds.dataset(timeseries_path, format="parquet")

    filt = (
        (ds.field("CountyFIPS") == county_fips) &
        (ds.field("RecordDateTime") >= window_start.to_pydatetime()) &
        (ds.field("RecordDateTime") <= window_end.to_pydatetime())
    )

    table = dataset.to_table(
        filter=filt,
        columns=["RecordDateTime", "CountyFIPS", "OutageFraction", "CustomersTracked"]
    )

    if table.num_rows == 0:
        return pd.DataFrame(columns=["datetime", "CountyFIPS", "outageFraction", "customersTracked"])

    df = table.to_pandas()
    df = df.rename(columns={
        "RecordDateTime": "datetime",
        "OutageFraction": "outageFraction",
        "CustomersTracked": "customersTracked",
    })
    return df

rows = []
for _, evt in selected_events.iterrows():
    outage = load_outage_window(TIMESERIES_PATH, evt["geoid"], evt["window_start"], evt["window_end"])
    if outage.empty:
        print(f"No outage rows found for {evt['event_id']} / {evt['geoid']}")
        continue

    outage = outage.copy()
    outage["event_id"] = evt["event_id"]
    outage["storm"] = evt["storm"]
    outage["geoid"] = evt["geoid"]
    outage["event_start"] = evt["event_start"]
    outage["duration_hours"] = evt["duration_hours"]
    outage["window_start"] = evt["window_start"]
    outage["window_end"] = evt["window_end"]
    rows.append(outage)

outage_long = pd.concat(rows, ignore_index=True)
print("Outage window rows:", len(outage_long))
outage_long.head()


Outage window rows: 580


,outageFraction,customersTracked,event_id,storm,geoid,event_start,duration_hours,window_start,window_end
0,0.0,785.0,event_002,2017242N16333,12067,2017-09-09 20:00:00,191,2017-09-09 08:00:00,2017-09-18 07:00:00
1,0.0,785.0,event_002,2017242N16333,12067,2017-09-09 20:00:00,191,2017-09-09 08:00:00,2017-09-18 07:00:00
2,0.0,785.0,event_002,2017242N16333,12067,2017-09-09 20:00:00,191,2017-09-09 08:00:00,2017-09-18 07:00:00
3,0.0,785.0,event_002,2017242N16333,12067,2017-09-09 20:00:00,191,2017-09-09 08:00:00,2017-09-18 07:00:00
4,0.0,785.0,event_002,2017242N16333,12067,2017-09-09 20:00:00,191,2017-09-09 08:00:00,2017-09-18 07:00:00


In [12]:
# Use lowercase 'h' to avoid the warning.
outage_long["datetime"] = pd.to_datetime(outage_long["datetime"]).dt.floor("h")
weather["datetime"] = pd.to_datetime(weather["datetime"]).dt.floor("h")

key_cols = ["event_id", "geoid", "datetime"]

print("Duplicate keys in outage_long:", outage_long.duplicated(key_cols).sum())
print("Duplicate keys in weather:", weather.duplicated(key_cols).sum())

dup_outage = outage_long[outage_long.duplicated(key_cols, keep=False)].sort_values(key_cols)
dup_weather = weather[weather.duplicated(key_cols, keep=False)].sort_values(key_cols)

print("Outage duplicate rows:")
print(dup_outage.head(20))

print("Weather duplicate rows:")
print(dup_weather.head(20))

Duplicate keys in outage_long: 577
Duplicate keys in weather: 0
Outage duplicate rows:
     index  outageFraction  customersTracked   event_id          storm  geoid         event_start  duration_hours            datetime  \
216    216        0.000000            5563.0  event_001  2017242N16333  12075 2017-09-09 22:00:00             190 2017-09-09 10:00:00   
217    217        0.000000            5563.0  event_001  2017242N16333  12075 2017-09-09 22:00:00             190 2017-09-09 10:00:00   
218    218        0.000000            5563.0  event_001  2017242N16333  12075 2017-09-09 22:00:00             190 2017-09-09 10:00:00   
219    219        0.000000            5563.0  event_001  2017242N16333  12075 2017-09-09 22:00:00             190 2017-09-09 10:00:00   
220    220        0.000000            5563.0  event_001  2017242N16333  12075 2017-09-09 22:00:00             190 2017-09-09 10:00:00   
221    221        0.000000            5563.0  event_001  2017242N16333  12075 2017-09-09 22

started messing with things around here ~

In [ ]:
# Dual-axis plot: outage fraction on the left, gust on the right.
def plot_event_dual_axis(df, event_id):
    d = df[df["event_id"] == event_id].sort_values("datetime").copy()
    if d.empty:
        print(f"No rows for {event_id}")
        return

    fig, ax1 = plt.subplots(figsize=(12, 4))

    ax1.plot(d["datetime"], d["outageFraction"], label="Outage fraction", linewidth=2)
    ax1.set_ylabel("Outage fraction")
    ax1.set_xlabel("Time")

    ax2 = ax1.twinx()
    ax2.plot(d["datetime"], d["gust_mps"], label="Wind gust (m/s)", linewidth=2)
    ax2.plot(d["datetime"], d["wind_speed_mps"], label="Wind speed (m/s)", linewidth=1.5, linestyle="--", alpha=0.8)
    ax2.set_ylabel("Wind (m/s)")

    event_start = d["event_start"].iloc[0]
    ax1.axvline(event_start, color="black", linestyle=":", linewidth=1)

    locator = AutoDateLocator()
    formatter = ConciseDateFormatter(locator)
    ax1.xaxis.set_major_locator(locator)
    ax1.xaxis.set_major_formatter(formatter)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

    title = f"{event_id} | county {d['geoid'].iloc[0]} | storm {d['storm'].iloc[0]}"
    ax1.set_title(title)
    plt.tight_layout()
    plt.show()

for event_id in selected_events["event_id"]:
    plot_event_dual_axis(merged, event_id)


### Optional next checks

- Swap `gust_mps` for `precip_mm` or `pressure_hpa` on the right axis.
- Increase `N_EVENTS` if the first few look sensible.
- If the alignment is poor, check the event windows before moving to modelling.
